# Reuse one source capture for two studies

## Objective

Read and save the source once, then create two studies from those saved rows.
We will change the original source and check that neither study uses the changed values.
We will also change a selection date and see the second study make its own eligibility decision.

This lesson uses files and the default `hash` mode. A hash is a fingerprint used to detect changes to saved contents. No database account is needed.
Use a fresh Python kernel and run the cells in order. The example has two products and
17 observations: A has days 1 to 9; B has days 2 to 9. No rows are removed from the raw capture.

## Step 1: create a lesson folder and start Spark

A registry records saved studies, their IDs and where their inputs live. Each run gets its own folder outside the repository. Spark reads the small example and
performs the same preparation checks that it would use for a larger study.

In [ ]:
import json
import os
import uuid
from datetime import date
from pathlib import Path

from autoforecast.study_registration import StudyRegistry, register_study
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Keep this lesson's files separate from earlier runs.
base = Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", "~/autoforecast-tutorials")).expanduser()
folder = base / ("shared-source-" + uuid.uuid4().hex[:8])
folder.mkdir(parents=True)

# Use two local workers for this small example.
spark = (SparkSession.builder.master("local[2]").appName("shared-source-lesson")
         .config("spark.sql.shuffle.partitions", "2").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
registry = StudyRegistry(folder / "registry", spark=spark)
print("Lesson folder:", folder)

**Understand the output:** the printed path is where this lesson will save its data,
capture and studies. Keep it if you want to reopen them later.

## Step 2: write the source data

Product B starts one day after A. That small difference will let us see why a new
selection date needs a new eligibility check, even when the input rows are unchanged.

In [ ]:
# Make all 17 source rows, with the day number as the observed value.
rows = [(product, date(2024, 1, day), float(day))
        for product in ("A", "B")
        for day in range(1 if product == "A" else 2, 10)]
source = spark.createDataFrame(rows, "item string, day date, value double")
source_path = folder / "source.parquet"
source.write.parquet(str(source_path))
print("Source observations:", source.count())
source.groupBy("item").agg(F.min("day").alias("first_day"), F.count("*").alias("rows")).show()

**Understand the output:** there are 17 observations. A starts on 1 January and has
9 rows. B starts on 2 January and has 8 rows. These are the complete raw inputs.

## Step 3: describe the study

The dataset section names the ID, date and value columns. The recipe chooses these windows:

| Stage | Dates | Products |
|---|---|---|
| Training for model selection | Through 5 January | A and B |
| Validation | 6 and 7 January | A and B |
| Refit before the final test | Through 7 January | A and B |
| Final test | 8 and 9 January | A and B |

`validation_origins` marks the end of initial training and also determines which products are eligible. Both products exist by 5 January. B has no observation on 1 January; this example does not invent one. The source section points to the file we just wrote.


The configuration has three parts:

- **Columns:** `item` identifies a product, `day` is its date, and `value` is the observation. `D` means daily. `observed_nonnegative` means the observed values must be zero or positive; it does not restrict model predictions.
- **Study rules:** the recipe records the rules for the study. Validation checks forecasts on dates held out from initial training. Refitting trains the final model again on the later history before testing. `new_series_policy="error"` stops registration if a product has no history by the first selection cutoff. `history_policy="fixed"` refuses conflicting observations where training inputs overlap.
- **Files:** `source` points to the original data file. `storage` names the registry folder for saved studies. Each `schema_version` identifies a configuration format, not a source-table version or capture time.

Registration can add missing dates within each product's required history. `history_missing="zero"` fills missing targets with zero when preparing initial training and refit data, including those added dates. It does not add history before a product starts, so B still has no 1 January row. Missing validation and test actuals follow the separate `actual_missing` rule. Here, `mask` leaves those missing actuals out of scoring.


In [ ]:
# Describe the columns and the dates the studies may use.
settings = {
    # Configuration format version, not a version of the source table.
    "schema_version": 1,
    # Name the product, date and observed-value columns.
    "dataset": {"schema_version": 1, "dataset": {"id": "two_products", "panel": {
        "series_col": "item", "time_col": "day", "target_col": "value",
        # Daily observations; a one-day baseline; zero or positive actual values.
        "frequency": "D", "season_length": 1, "target_estimand": "observed_nonnegative",
    }}},
    # Choose the training, validation, refit and final-test windows.
    "recipe": {
        "schema_version": 2, "source_keys": ["item"], "training_start": "2024-01-01",
        "validation_origins": ["2024-01-05"], "validation_horizon": 2,
        "final_train_cutoff": "2024-01-07", "final_test_horizon": 2,
        # Refuse conflicting history and products that start after the first cutoff.
        "history_policy": "fixed", "new_series_policy": "error", "target_unit": "units",
        # No extra features; score the average absolute forecast error.
        "feature_availability": {}, "metric": "mae",
        # Fill missing fit targets, including calendar dates added after a product starts.
        "history_missing": "zero",
        # Leave missing actual values out of scoring.
        "actual_missing": "mask",
    },
    # Read the original data here; save studies in a separate registry folder.
    "source": {"kind": "parquet", "path": str(source_path.absolute())},
    "storage": {"root": str((folder / "registry").absolute())},
}
# Step 4 reads this file to create a capture.
# Step 6 replaces the source path with the saved capture ID.
config = folder / "study.json"
config.write_text(json.dumps(settings, indent=2))
print("Configuration saved:", config)


**Understand the output:** this is a configuration file, not a trained model.
The capture will save raw inputs covering this declared date scope. Later studies can use
a smaller scope; a request beyond it needs a new capture.

## Step 4: capture the raw inputs once

A source capture saves the rows before recipe-dependent preparation. A study also saves
its chosen series, dates and evaluation rules. We create the capture first so both studies
can start from exactly the same raw rows.

In [ ]:
# Save the raw source and keep the returned ID for later registrations.
capture_id = registry.capture_sources(config)
print("Source capture ID:", capture_id)

**Understand the output:** the long ID identifies the saved capture. No forecast has
been fitted or evaluated. The ID identifies the saved inputs and their recorded settings.

## Step 5: change the old source

We deliberately replace every value in the original source with 999. This is a test:
a study that reopens the saved capture must still see the original values.

In [ ]:
# Replace only this lesson's original source, leaving its saved capture alone.
source.withColumn("value", F.lit(999.0)).write.mode("overwrite").parquet(str(source_path))
print("Live source value:", spark.read.parquet(str(source_path)).first()["value"])

**Understand the output:** the live source now contains 999. The capture has its own
saved inputs, so changing the source does not update the capture.

## Step 6: register the first study from the capture

Replace the source path with the capture ID. Registration reads the saved rows and applies the recipe. It does not reopen the old source file.

We check every product, date and value in all four stages. The expected counts are 9 training, 4 validation, 13 refit and 4 test rows. These windows overlap, so their counts do not add up to the 17 source observations. Collecting all rows is suitable only for this tiny example.


In [ ]:
# Tell registration to reuse the saved raw inputs.
settings["source"] = {"kind": "capture", "capture_id": capture_id}
config.write_text(json.dumps(settings, indent=2))
first_id = register_study(config, registry=registry)

# Check every row in a small stage, including any repeated observations.
def check_rows(frame, products, first_day, last_day):
    expected = sorted(row for row in rows
                      if row[0] in products and first_day <= row[1].day <= last_day)
    actual = sorted(tuple(row) for row in frame.select("item", "day", "value").collect())
    assert actual == expected, "Saved rows differ from the original toy data"
    return len(actual)

# Read each stage inside the study's read handle.
with registry.load_study(first_id) as first:
    assert first.n_series == 2
    for stage, start, end in [("training", 1, 5), ("validation", 6, 7),
                              ("refit", 1, 7), ("testing", 8, 9)]:
        count = check_rows(getattr(first, stage + "_data"), {"A", "B"}, start, end)
        print(stage, "original rows checked:", count)


**Understand the output:** the four counts are 9, 4, 13 and 4. Both products are included. Every stage matches its original rows; none uses the changed source values. The comparisons check the values too, not just these counts.


## Step 7: give the second study an earlier training cutoff

Move `validation_origins` to 1 January. This ends initial training on 1 January and moves validation to 2 and 3 January. The final refit still ends on 7 January, and the final test still uses 8 and 9 January.

A exists on the earlier cutoff date; B starts later. We choose the policy that excludes products arriving after that cutoff. Only A remains in all four stages. AutoForecast must recalculate this decision from the raw capture instead of copying the first study's result.


In [ ]:
# Change this child's cutoff and eligibility rule, keeping the same raw capture.
settings["recipe"]["validation_origins"] = ["2024-01-01"]
settings["recipe"]["new_series_policy"] = "exclude_after_first_origin"
config.write_text(json.dumps(settings, indent=2))
second_id = register_study(config, registry=registry)

# Check A's complete original rows in each new stage window.
with registry.load_study(second_id) as second:
    assert second.n_series == 1
    for stage, start, end in [("training", 1, 1), ("validation", 2, 3),
                              ("refit", 1, 7), ("testing", 8, 9)]:
        count = check_rows(getattr(second, stage + "_data"), {"A"}, start, end)
        print(stage, "original rows checked:", count)
assert first_id != second_id
print("Different study IDs:", first_id != second_id)


**Understand the output:** the second study has 1 training row, 2 validation rows, 7 refit rows and 2 test rows, all for A. Every value matches the original data. The IDs differ because the studies have different rules and populations. Reuse saved a source read; it did not reuse the first study's eligibility decision.


## Step 8: close the lesson

Close owned connections and stop the local Spark session. The saved files remain in the
printed lesson folder so you can inspect or reopen them. A table capture cannot be released while
a registered dependent study still needs it.

In [ ]:
# Finish this local session without deleting its saved work.
registry.close()
spark.stop()
print("Saved lesson folder:", folder)

**Understand the output:** the saved files remain; the local session has ended.
For protected table captures, use the setup and recovery instructions in the tutorial guide.
Choose the same storage mode when creating a capture and registering its child studies.